In [33]:
from crewai_tools import tool
import yfinance as yf
import pandas as pd
from typing import List, Dict

@tool("Stock News")
def stock_news(ticker: str) -> List[Dict]:
    """
    Retrieves the latest news about a stock.
    
    Args:
        ticker (str): The stock ticker symbol, e.g., 'AAPL' for Apple, 'NET' for Cloudflare.
    
    Returns:
        list: A list of dictionaries containing news articles related to the stock.
    """
    ticker = yf.Ticker(ticker)
    return ticker.news

@tool("Stock Price")
def stock_price(ticker: str) -> pd.DataFrame:
    """
    Retrieves the historical stock prices for the past month.
    
    Args:
        ticker (str): The stock ticker symbol, e.g., 'AAPL' for Apple, 'NET' for Cloudflare.
    
    Returns:
        pandas.DataFrame: A DataFrame containing the historical stock prices.
    """
    ticker = yf.Ticker(ticker)
    return ticker.history(period="1mo")

@tool("Income Statement")
def income_stmt(ticker: str) -> pd.DataFrame:
    """
    Retrieves the income statement for a stock.
    
    Args:
        ticker (str): The stock ticker symbol, e.g., 'AAPL' for Apple, 'NET' for Cloudflare.
    
    Returns:
        pandas.DataFrame: A DataFrame containing the income statement.
    """
    ticker = yf.Ticker(ticker)
    return ticker.income_stmt

@tool("Balance Sheet")
def balance_sheet(ticker: str) -> pd.DataFrame:
    """
    Retrieves the balance sheet for a stock.
    
    Args:
        ticker (str): The stock ticker symbol, e.g., 'AAPL' for Apple, 'NET' for Cloudflare.
    
    Returns:
        pandas.DataFrame: A DataFrame containing the balance sheet.
    """
    ticker = yf.Ticker(ticker)
    return ticker.balance_sheet

@tool("Insider Transactions")
def insider_transactions(ticker: str) -> pd.DataFrame:
    """
    Retrieves insider transactions for a stock.
    
    Args:
        ticker (str): The stock ticker symbol, e.g., 'AAPL' for Apple, 'NET' for Cloudflare.
    
    Returns:
        pandas.DataFrame: A DataFrame containing insider transactions.
    """
    ticker = yf.Ticker(ticker)
    return ticker.insider_transactions


In [34]:
from crewai_tools import ScrapeWebsiteTool

scrape_tool = ScrapeWebsiteTool()

In [35]:
from crewai import Agent

researcher = Agent(
    role="Researcher",
    goal="""
        Gather and interpret vast amounts of data to
        provide a comprehensive overview of the sentiment
        and news surrounding a stock.
    """,
    backstory="""
        You're skilled in gathering and interpreting data
        from various sources. You read each data source
        carefully and extract the most important information.
        Your insights are crucial for making informed
        investment decisions.
    """,
    tools=[
		scrape_tool,
        stock_news
	]
)

technical_analyst = Agent(
    role="Technical Analyst",
    goal="""
        Analyze the movements of a stock and provide
        insights on trends, entry points, resistance, and
        support levels.
    """,
    backstory="""
        An expert in technical analysis, you're known
        for your ability to predict stock prices.
        You provide valuable insights to your clients.
    """,
    tools=[
        stock_price
    ]
)

financial_analyst = Agent(
    role="Financial Analyst",
    goal="""
        Use financial statements, insider trading data,
        and other metrics to evaluate a stock's financial
        health and performance.
    """,
    backstory="""
        You're a very experienced investment advisor
        who looks at a company's financial health,
        market sentiment, and qualitative data to
        make informed recommendations.
    """,
    tools=[
        income_stmt,
        balance_sheet,
        insider_transactions
    ]
)

hedge_fund_manager = Agent(
    role="Hedge Fund Manager",
    goal="""
        Manage a portfolio of stocks and make investment
        decisions to maximize returns using insights
        from financial analysts and researchers.
    """,
    backstory="""
        You're a seasoned hedge fund manager with a proven
        track record of making profitable investments.
        You always impress your clients.
    """
)


In [36]:
from crewai import Task

research = Task(
    description="""
        Gather and analyze the latest news and
        market sentiment surrounding {company}'s stock.
        Provide a summary of the news and any notable
        shifts in sentiment.
    """,
    expected_output="""
        Your final answer **MUST** be a detailed
        summary of the news and market sentiment
        surrounding the stock.
    """,
    agent=researcher,
)

technical_analysis = Task(
    description="""
        Conduct a technical analysis of the {company}
        stock price movements and identify
        key support and resistance levels and chart patterns.
    """,
    expected_output="""
        Your final answer **MUST** be a report
        with potential entry points, price targets,
        and any other relevant information.
    """,
    agent=technical_analyst,
)

financial_analysis = Task(
    description="""
        Analyze {company}'s financial statements,
        balance sheet, insider trading data,
        and other metrics to evaluate {company}'s
        financial health and performance.
    """,
    expected_output="""
        Your final answer **MUST** be a report with
        an overview of {company}'s revenue, earnings,
        cash flow, and other key financial metrics.
    """,
    agent=financial_analyst,
)

investment_recommendation = Task(
    description="""
        Based on the research, technical analysis, and
        financial analysis reports, provide a detailed
        investment recommendation for {company} stock.
    """,
    expected_output="""
        Your final answer **MUST** be a detailed
        recommendation to BUY, SELL, or HOLD the stock.
        Provide a clear rationale for your recommendation.
    """,
    agent=hedge_fund_manager,
    context=[
        research,
        technical_analysis,
        financial_analysis
    ],
    output_file="investment_recommendation.md"
)

In [37]:
from crewai import Crew

crew = Crew(
    tasks=[
        research,
        technical_analysis,
        financial_analysis,
        investment_recommendation
    ],
    agents=[
        researcher,
        technical_analyst,
        financial_analyst,
        hedge_fund_manager,
    ],
    verbose=2,
)

2024-08-06 18:02:08,168 - 8054529088 - __init__.py-__init__:531 - WARNING: Overriding of current TracerProvider is not allowed


In [40]:
result = crew.kickoff(
    inputs={
        "company": "3655.T",
    }
)

 [2024-08-06 18:04:42][DEBUG]: == Working Agent: Researcher
 [2024-08-06 18:04:42][INFO]: == Starting Task: 
        Gather and analyze the latest news and
        market sentiment surrounding 3655.T's stock.
        Provide a summary of the news and any notable
        shifts in sentiment.
    
 

[{'uuid': '1d65ccfe-73d2-324c-8118-83252e5591fb', 'title': "No Costco Membership? No Problem. Here's How You Can Still Shop", 'publisher': 'Motley Fool', 'link': 'https://finance.yahoo.com/m/1d65ccfe-73d2-324c-8118-83252e5591fb/no-costco-membership%3F-no.html', 'providerPublishTime': 1722934829, 'type': 'STORY', 'thumbnail': {'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/TczPUzR0pPX7MLSntIVgeg--~B/aD01Mjg7dz03OTM7YXBwaWQ9eXRhY2h5b24-/https://media.zenfs.com/en/motleyfool.com/0c6487ad884b05294b9761d0de1ab5ac', 'width': 793, 'height': 528, 'tag': 'original'}, {'url': 'https://s.yimg.com/uu/api/res/1.2/H4.SqSQVaVKvgDm8loV3BA--~B/Zmk9ZmlsbDtoPTE0MDtweW9mZj0wO3c9MTQwO2FwcGlkPXl0YWNoe